In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np
from tqdm import tqdm
import copy
import time

In [ ]:
BASE_PATH = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR  = os.path.join(BASE_PATH, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 5        
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Using device: {DEVICE}")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)

train_size = int(0.85 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

val_dataset.dataset.transform = val_test_transform
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

In [ ]:
targets = [full_train_dataset.targets[i] for i in train_dataset.indices]
class_counts = np.bincount(targets)
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = class_weights.to(DEVICE)

print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights.cpu().numpy()}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

# Freeze all feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_features, 2)
)
model = model.to(DEVICE)

print("→ Backbone is frozen. Only classifier will be trained.")

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="weighted")
    return epoch_loss, epoch_acc, epoch_f1

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="weighted")
    epoch_auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return epoch_loss, epoch_acc, epoch_f1, epoch_auc, cm

In [ ]:
best_val_f1 = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
epoch3_model_wts = None

print("\nStarting DenseNet121 (Frozen Backbone) training...\n")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print("-" * 50)

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )
    val_loss, val_acc, val_f1, val_auc, val_cm = evaluate(
        model, val_loader, criterion, DEVICE
    )

    scheduler.step()

    print(f"Train → Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"Val Confusion Matrix:\n{val_cm}")

    # Save Epoch 3 model
    if epoch == 3:
        epoch3_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "densenet121_epoch3.pth")
        print("→ Model at Epoch 3 saved as 'densenet121_epoch3.pth'")

    # Track best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "densenet121_frozen_best.pth")
        print("→ Best frozen model saved!")

    print()

total_time = time.time() - start_time
print(f"Training finished in {total_time/60:.1f} minutes. Best Val F1: {best_val_f1:.4f}")

In [ ]:
print("\n" + "="*60)
print("TEST SET EVALUATION - Best Frozen Model")
print("="*60)

model.load_state_dict(best_model_wts)
test_loss, test_acc, test_f1, test_auc, test_cm = evaluate(model, test_loader, criterion, DEVICE)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test F1-Score : {test_f1:.4f}")
print(f"Test AUC      : {test_auc:.4f}")
print(f"\nConfusion Matrix:\n{test_cm}")

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nClassification Report (Best Frozen Model):")
print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
if epoch3_model_wts is not None:
    print("\n" + "="*60)
    print("TEST SET EVALUATION - Epoch 3 Model")
    print("="*60)

    model.load_state_dict(epoch3_model_wts)
    test_loss, test_acc, test_f1, test_auc, test_cm = evaluate(model, test_loader, criterion, DEVICE)

    print(f"Test Loss     : {test_loss:.4f}")
    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"Test F1-Score : {test_f1:.4f}")
    print(f"Test AUC      : {test_auc:.4f}")
    print(f"\nConfusion Matrix:\n{test_cm}")

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print("\nClassification Report (Epoch 3 Model):")
    print(classification_report(all_labels, all_preds, target_names=['NORMAL', 'PNEUMONIA']))

print("\nModels saved:")
print(" - densenet121_epoch3.pth")
print(" - densenet121_frozen_best.pth")